# Projet actuel — classification locale de bâtiments

Ce notebook remplace la lecture pédagogique de l'ancien `cdt05.ipynb` pour qu'elle corresponde au projet actuel.

Objectif : expliquer et visualiser le fonctionnement actuel du projet sans réentraîner les modèles, sans modifier les fichiers du projet et sans dépendre de l'ancien dataset.

Le projet classe des images de bâtiments en trois styles :

| Label | Classe |
|---:|---|
| 0 | Art déco |
| 1 | Art nouveau |
| 2 | Gothique |

Les algorithmes de machine learning sont implémentés en C++ dans `ml_library`. Python sert ici uniquement à documenter, lire les rapports JSON et afficher des graphiques.

## Ce qui change par rapport à l'ancien notebook

L'ancien notebook était utile pour présenter les premiers essais, mais il ne représente plus l'état réel du projet.

| Ancienne version | Version actuelle |
|---|---|
| Dataset de 2 548 images | Dataset sans doublon de 1 852 images |
| Sous-échantillon de 90 images | Modèles entraînés sur le CSV canonique actuel |
| Images 16×16 | Images 32×32 |
| 256 features | 1 024 features |
| Pas de manifest | `models/models_manifest.json` pilote les modèles disponibles |
| Pas de MLP v2 / RBF v2 | MLP v2 recommandé, RBF v2 corrigé si présent |
| Accuracy seule | Accuracy, balanced accuracy et matrices de confusion |

Ce notebook lit donc les fichiers déjà générés par le projet : manifest et rapports JSON. Il ne relance pas l'entraînement.

In [ ]:
from pathlib import Path
import json
import math

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
PROJECT = ROOT / "projet_rendu_2_ml"
MODELS_DIR = PROJECT / "models"
REPORTS_DIR = MODELS_DIR / "reports"
TUNING_DIR = REPORTS_DIR / "tuning"
MANIFEST_PATH = MODELS_DIR / "models_manifest.json"

CLASS_NAMES = {
    0: "Art déco",
    1: "Art nouveau",
    2: "Gothique",
}

print("Racine détectée :", ROOT)
print("Projet :", PROJECT)
print("Manifest :", MANIFEST_PATH)

## Validation légère des fichiers

Cette cellule vérifie seulement que les fichiers nécessaires existent : manifest, modèles référencés et rapports JSON.

Elle ne compile rien, n'entraîne rien, ne supprime rien et ne génère aucun modèle.

In [ ]:
def load_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def status_line(ok, message):
    prefix = "OK" if ok else "WARNING"
    print(f"[{prefix}] {message}")

status_line(MANIFEST_PATH.exists(), f"manifest : {MANIFEST_PATH}")
manifest = load_json(MANIFEST_PATH) if MANIFEST_PATH.exists() else {"models": []}

for model in manifest.get("models", []):
    model_path = MODELS_DIR / model.get("file", "")
    report_path = REPORTS_DIR / f"{model.get('id', '')}.json"
    status_line(model_path.exists(), f"modèle {model.get('id')} : {model_path.name}")
    status_line(report_path.exists(), f"rapport {model.get('id')} : {report_path.name}")

for optional_report in ["mlp_v1.json", "rbf_v1.json", "tuning_summary.json"]:
    folder = TUNING_DIR if optional_report == "tuning_summary.json" else REPORTS_DIR
    path = folder / optional_report
    status_line(path.exists(), f"rapport de comparaison : {path.relative_to(PROJECT) if path.exists() else path}")

## Dataset actuel

Le dataset actuel est le dataset local sans doublon :

`projet_rendu_2_ml/data/Dataset final-20260702T055807Z-3-001/Dataset final sans doublon`

Distribution connue :

| Classe | Label | Images |
|---|---:|---:|
| Art déco | 0 | 554 |
| Art nouveau | 1 | 745 |
| Gothique | 2 | 553 |
| **Total** |  | **1 852** |

Le CSV canonique associé est `projet_rendu_2_ml/data/batiments_3_classes.csv`.

Chaque ligne contient 1 024 valeurs numériques, puis le label de classe.

In [ ]:
dataset_counts = {
    "Art déco": 554,
    "Art nouveau": 745,
    "Gothique": 553,
}

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(dataset_counts.keys(), dataset_counts.values(), color=["#5B8FF9", "#61DDAA", "#65789B"])
ax.set_title("Distribution du dataset actuel")
ax.set_ylabel("Nombre d'images")
ax.set_ylim(0, max(dataset_counts.values()) * 1.2)
for index, value in enumerate(dataset_counts.values()):
    ax.text(index, value + 10, str(value), ha="center")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

## Prétraitement image

Le même prétraitement est utilisé pour générer le CSV et pour traiter les images envoyées à l'interface web :

1. correction de l'orientation avec `ImageOps.exif_transpose` ;
2. conversion en niveaux de gris ;
3. redimensionnement en 32×32 ;
4. interpolation `Image.Resampling.LANCZOS` ;
5. conversion en nombres flottants ;
6. normalisation dans `[0, 1]` ;
7. aplatissement ligne par ligne ;
8. vérification stricte de 1 024 valeurs.

Important : ce prétraitement n'est pas du machine learning. Il transforme seulement une image en vecteur numérique compatible avec les modèles C++.

## Architecture C++ / Python

Le projet sépare clairement l'entraînement, la prédiction et l'interface web.

### Entraînement hors ligne

```text
CSV canonique
→ train_cli.exe
→ ml_api.cpp
→ ml_library
→ modèle sauvegardé + rapport JSON
→ models_manifest.json
```

### Prédiction via l'application web

```text
Navigateur
→ Flask
→ prétraitement image en 1 024 valeurs
→ predict_cli.exe
→ ml_load
→ ml_predict / ml_predict_with_score
→ JSON
→ affichage web
```

Flask ne réentraîne pas les modèles. Il lit le manifest, choisit un modèle déjà sauvegardé, prépare les features, appelle `predict_cli.exe`, puis affiche le résultat.

## Fonctions utilitaires pour lire les résultats

Les cellules suivantes lisent le manifest et les rapports JSON. Si la balanced accuracy n'est pas présente dans un rapport, elle est calculée uniquement à partir de la matrice de confusion déjà enregistrée.

In [ ]:
def confusion_matrix(report_or_model):
    matrix = report_or_model.get("confusion_matrix")
    return np.array(matrix, dtype=float) if matrix is not None else None

def balanced_accuracy_from_matrix(matrix):
    matrix = np.array(matrix, dtype=float)
    recalls = []
    for class_index in range(matrix.shape[0]):
        total = matrix[class_index, :].sum()
        recalls.append(matrix[class_index, class_index] / total if total else 0.0)
    return float(np.mean(recalls))

def prediction_distribution(matrix):
    matrix = np.array(matrix, dtype=float)
    return matrix.sum(axis=0).astype(int).tolist()

def accuracy_percent(value):
    return f"{value * 100:.2f} %" if isinstance(value, (int, float)) else "n/a"

def report_for_model_id(model_id):
    path = REPORTS_DIR / f"{model_id}.json"
    return load_json(path) if path.exists() else None

def model_rows_from_manifest(manifest):
    rows = []
    for model in manifest.get("models", []):
        matrix = model.get("confusion_matrix")
        balanced = model.get("balanced_accuracy")
        if balanced is None and matrix is not None:
            balanced = balanced_accuracy_from_matrix(matrix)
        rows.append({
            "id": model.get("id"),
            "display_name": model.get("display_name"),
            "algorithm": model.get("algorithm"),
            "file": model.get("file"),
            "score_type": model.get("score_type"),
            "train_accuracy": model.get("train_accuracy"),
            "test_accuracy": model.get("test_accuracy"),
            "balanced_accuracy": balanced,
            "parameters": model.get("parameters", {}),
            "confusion_matrix": matrix,
            "model_size_bytes": model.get("model_size_bytes"),
            "duration_seconds": model.get("duration_seconds"),
        })
    return rows

def print_table(rows, headers):
    widths = [len(header) for header in headers]
    body = []
    for row in rows:
        line = [str(row.get(header, "")) for header in headers]
        widths = [max(widths[i], len(line[i])) for i in range(len(headers))]
        body.append(line)
    print(" | ".join(headers[i].ljust(widths[i]) for i in range(len(headers))))
    print("-+-".join("-" * widths[i] for i in range(len(headers))))
    for line in body:
        print(" | ".join(line[i].ljust(widths[i]) for i in range(len(headers))))

models = model_rows_from_manifest(manifest)
print(f"Modèles proposés par le manifest : {len(models)}")

## Modèles disponibles via `models_manifest.json`

Le manifest est la source utilisée par l'application web pour savoir quels modèles afficher.

Les scores ne veulent pas tous dire la même chose :

- Perceptron et SVM : `margin` ;
- MLP et RBF : `best_ovr_probability` ;
- les scores sont comparables à l'intérieur d'un même modèle, pas entre algorithmes différents.

In [ ]:
table_rows = []
for model in models:
    table_rows.append({
        "id": model["id"],
        "algo": model["algorithm"],
        "score": model["score_type"],
        "train": accuracy_percent(model["train_accuracy"]),
        "test": accuracy_percent(model["test_accuracy"]),
        "balanced": accuracy_percent(model["balanced_accuracy"]),
        "fichier": model["file"],
    })

print_table(table_rows, ["id", "algo", "score", "train", "test", "balanced", "fichier"])

## Résultats actuels des modèles

Les graphiques suivants comparent uniquement les modèles actuellement référencés par le manifest.

Le modèle recommandé actuellement est **MLP v2**, car il obtient la meilleure accuracy test globale dans les rapports disponibles.

In [ ]:
labels = [model["id"] for model in models]
test_values = [model["test_accuracy"] for model in models]
balanced_values = [model["balanced_accuracy"] for model in models]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

axes[0].bar(labels, test_values, color="#5B8FF9")
axes[0].set_title("Accuracy test")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(labels, balanced_values, color="#61DDAA")
axes[1].set_title("Balanced accuracy")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=20)

for ax, values in zip(axes, [test_values, balanced_values]):
    for index, value in enumerate(values):
        ax.text(index, value + 0.02, f"{value*100:.1f}%", ha="center")

plt.tight_layout()
plt.show()

## Matrices de confusion

Dans ces matrices, les lignes représentent la vraie classe et les colonnes la classe prédite.

Une bonne matrice a des valeurs fortes sur la diagonale. Les valeurs hors diagonale représentent les erreurs.

In [ ]:
def plot_confusion(ax, matrix, title):
    matrix = np.array(matrix, dtype=int)
    image = ax.imshow(matrix, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Classe prédite")
    ax.set_ylabel("Classe réelle")
    ax.set_xticks(range(3), [CLASS_NAMES[i] for i in range(3)], rotation=35, ha="right")
    ax.set_yticks(range(3), [CLASS_NAMES[i] for i in range(3)])
    for row in range(matrix.shape[0]):
        for col in range(matrix.shape[1]):
            ax.text(col, row, str(matrix[row, col]), ha="center", va="center", color="black")
    return image

cols = 2
rows = math.ceil(len(models) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(12, 5 * rows))
axes = np.array(axes).reshape(-1)

for ax, model in zip(axes, models):
    plot_confusion(ax, model["confusion_matrix"], model["id"])

for ax in axes[len(models):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## Analyse MLP v2

MLP v2 est le modèle recommandé actuellement.

Par rapport à MLP v1, il utilise plus de capacité et de meilleures valeurs de paramètres :

- MLP v1 : 8 epochs, learning rate 0,03, 16 neurones cachés ;
- MLP v2 : 12 epochs, learning rate 0,02, 32 neurones cachés.

Cette version améliore l'accuracy test tout en restant dans l'architecture C++ existante.

In [ ]:
mlp_reports = []
for report_name in ["mlp_v1", "mlp_v2"]:
    report = report_for_model_id(report_name)
    if report:
        matrix = report["confusion_matrix"]
        mlp_reports.append({
            "id": report_name,
            "test_accuracy": report["test_accuracy"],
            "balanced_accuracy": balanced_accuracy_from_matrix(matrix),
            "matrix": matrix,
            "parameters": report["parameters"],
        })

print_table([
    {
        "id": report["id"],
        "test": accuracy_percent(report["test_accuracy"]),
        "balanced": accuracy_percent(report["balanced_accuracy"]),
        "params": report["parameters"],
    }
    for report in mlp_reports
], ["id", "test", "balanced", "params"])

fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(len(mlp_reports))
ax.bar(x - 0.15, [r["test_accuracy"] for r in mlp_reports], width=0.3, label="accuracy test")
ax.bar(x + 0.15, [r["balanced_accuracy"] for r in mlp_reports], width=0.3, label="balanced accuracy")
ax.set_xticks(x, [r["id"] for r in mlp_reports])
ax.set_ylim(0, 1)
ax.set_title("Comparaison MLP v1 / MLP v2")
ax.legend()
plt.tight_layout()
plt.show()

## Pourquoi RBF v1 échouait

RBF v1 avait un comportement dégénéré : il prédisait presque toujours, et dans le rapport test exactement toujours, la classe **Gothique**.

La cause la plus probable vient de la combinaison suivante :

- les images sont représentées dans un espace de 1 024 dimensions ;
- `sigma = 1` était trop petit pour ces distances ;
- les activations `exp(-distance² / (2 sigma²))` devenaient proches de zéro ;
- les centres RBF n'apportaient presque plus d'information ;
- le biais one-vs-rest finissait par dominer ;
- la classe Gothique gagnait systématiquement.

RBF v2 corrige ce comportement si le manifest référence `rbf_v2` : sigma plus adapté, davantage de centres et sélection déterministe mieux répartie des centres. Il reste moins bon que MLP v2, mais il ne s'effondre plus sur une seule classe.

In [ ]:
rbf_reports = []
for report_name in ["rbf_v1", "rbf_v2"]:
    report = report_for_model_id(report_name)
    if report:
        matrix = report["confusion_matrix"]
        rbf_reports.append({
            "id": report_name,
            "test_accuracy": report["test_accuracy"],
            "balanced_accuracy": balanced_accuracy_from_matrix(matrix),
            "prediction_distribution": prediction_distribution(matrix),
            "matrix": matrix,
            "parameters": report["parameters"],
        })

print_table([
    {
        "id": report["id"],
        "test": accuracy_percent(report["test_accuracy"]),
        "balanced": accuracy_percent(report["balanced_accuracy"]),
        "prédictions [0,1,2]": report["prediction_distribution"],
        "params": report["parameters"],
    }
    for report in rbf_reports
], ["id", "test", "balanced", "prédictions [0,1,2]", "params"])

if rbf_reports:
    fig, axes = plt.subplots(1, len(rbf_reports), figsize=(6 * len(rbf_reports), 4))
    axes = np.array(axes).reshape(-1)
    for ax, report in zip(axes, rbf_reports):
        plot_confusion(ax, report["matrix"], report["id"])
    plt.tight_layout()
    plt.show()

## Résumé du tuning RBF

Le tuning RBF a été limité : il ne s'agit pas d'une grid search massive.

Le rapport `models/reports/tuning/tuning_summary.json` indique que RBF v2 a été retenu avec :

- `rbf_centers = 96` ;
- `rbf_sigma = 5` ;
- sélection des centres : `binary_stratified_even_spread` ;
- seed 42 ;
- split train/test inchangé.

La correction améliore surtout la balanced accuracy et évite la prédiction d'une seule classe.

In [ ]:
summary_path = TUNING_DIR / "tuning_summary.json"
if summary_path.exists():
    tuning_summary = load_json(summary_path)
    rbf_summary = tuning_summary.get("rbf", {})
    print("RBF tuning présent :", rbf_summary.get("tuned"))
    print("Paramètres retenus :", rbf_summary.get("selected_parameters"))
    print("Accuracy validation :", accuracy_percent(rbf_summary.get("selected_validation_accuracy")))
    print("Balanced accuracy validation :", accuracy_percent(rbf_summary.get("selected_validation_balanced_accuracy")))
    print("Accuracy test :", accuracy_percent(rbf_summary.get("selected_test_accuracy")))
    print("Balanced accuracy test :", accuracy_percent(rbf_summary.get("selected_test_balanced_accuracy")))
    print("Matrice test :", rbf_summary.get("selected_test_confusion_matrix"))
else:
    print("WARNING: tuning_summary.json absent")

## Comparaison des modèles

Lecture simple des résultats actuels :

- **MLP v2** est le meilleur choix pour la démonstration web actuelle ;
- **Perceptron v1** et **SVM v1** sont utiles comme baselines simples ;
- **RBF v2** est intéressant pédagogiquement, car il montre une correction réelle par rapport à RBF v1 ;
- aucun modèle n'atteint une performance très élevée, ce qui est cohérent avec un dataset visuel difficile et des modèles volontairement simples.

In [ ]:
parameter_rows = []
for model in models:
    params = model["parameters"]
    parameter_rows.append({
        "id": model["id"],
        "epochs": params.get("epochs"),
        "lr": params.get("learning_rate"),
        "hidden": params.get("hidden_size"),
        "centers": params.get("rbf_centers"),
        "sigma": params.get("rbf_sigma"),
        "lambda": params.get("svm_lambda"),
        "durée": f"{model['duration_seconds']:.2f}s" if model["duration_seconds"] else "n/a",
        "taille": f"{model['model_size_bytes']:,} octets" if model["model_size_bytes"] else "n/a",
    })

print_table(parameter_rows, ["id", "epochs", "lr", "hidden", "centers", "sigma", "lambda", "durée", "taille"])

## Limites actuelles

- Les modèles travaillent sur des images réduites en 32×32 niveaux de gris : beaucoup d'information visuelle est perdue.
- Les scores ne sont pas calibrés entre modèles différents.
- Le dataset reste de taille modérée pour une tâche visuelle.
- Les classes architecturales peuvent se ressembler fortement selon l'image.
- Les algorithmes sont volontairement simples et écrits en C++ pour le projet, sans framework ML externe.
- RBF v2 est amélioré par rapport à RBF v1, mais reste moins convaincant que MLP v2.

## Lecture du projet par rapport au syllabus

Le syllabus fourni parle d'un projet data complet : choix du dataset, import, parsing, transformations, statistiques descriptives, machine learning et data visualisation.

Dans notre rendu actuel, la partie machine learning et l'application locale sont très détaillées. La partie Hadoop/Spark n'est pas exécutée par ce notebook : elle doit être présentée comme hors périmètre de l'application C++/Flask actuelle, ou couverte dans un document/notebook séparé si elle est exigée explicitement à la soutenance.

| Étape du syllabus | Couverture dans ce projet |
|---|---|
| 0 — Choix du sujet et du dataset | Oui : classification d'images de bâtiments en 3 styles architecturaux |
| 1 — Importation du dataset sur HDFS | Non montré ici : l'application actuelle fonctionne en local, sans HDFS |
| 2 — Parsing des données | Oui : images transformées en CSV exploitable |
| 3 — Transformations, normalisation, dédoublonnage | Oui : dataset sans doublon, images 32×32, normalisation `[0,1]` |
| 4 — Statistiques descriptives | Oui : distribution des classes, tailles train/test, matrices de confusion |
| 5 — Machine Learning | Oui : Perceptron, MLP, RBF, SVM en C++ via `ml_library` |
| 6 — Data Viz | Oui : graphiques de distribution, accuracies, balanced accuracies et matrices de confusion |

Conclusion pédagogique : ce notebook est cohérent pour expliquer la partie ML locale du projet. Si le professeur attend une démonstration Hadoop/Spark, il faut l'annoncer séparément, car ce notebook ne lance volontairement ni HDFS ni Spark.

## Cohérence avec le notebook de cas de tests fourni

Le notebook `Cas de tests` fourni contient surtout des jeux synthétiques très simples pour réfléchir aux comportements attendus des algorithmes : séparations linéaires, XOR, croix, multi-classes et quelques formes de régression.

Ces cas ne remplacent pas le dataset bâtiments. Ils servent à expliquer la réflexion : avant d'appliquer un algorithme à des images réelles, on vérifie mentalement ou visuellement ce qu'il devrait faire sur des formes simples.

| Cas du notebook fourni | Intérêt pour notre projet |
|---|---|
| Linear Simple | Vérifie qu'un modèle linéaire peut séparer deux groupes simples |
| Linear Multiple | Vérifie une séparation linéaire avec plus de points |
| XOR | Montre une limite classique des modèles purement linéaires |
| Cross | Montre une frontière non linéaire plus difficile |
| Multi Linear 3 classes | Se rapproche de notre problème à 3 classes |
| Multi Cross | Montre une version multi-classe non linéaire, plus complexe |
| Régression | Utile pour la culture ML, mais hors application finale bâtiments qui est une classification |

Les cellules suivantes gardent donc des cas de tests visuels, pour représenter la démarche de réflexion, sans entraîner les modèles du projet et sans modifier aucun fichier.

In [ ]:
rng = np.random.default_rng(42)

def plot_case(ax, X, y, title):
    colors = np.array(["#5B8FF9", "#F6BD16", "#E8684A"])
    ax.scatter(X[:, 0], X[:, 1], c=colors[y], s=18, edgecolors="none")
    ax.set_title(title)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.grid(alpha=0.2)

# 1) Séparation linéaire simple
X_linear_simple = np.array([[1, 1], [2, 2], [3, 3], [1, 3], [2, 4], [3, 5]], dtype=float)
y_linear_simple = np.array([0, 0, 0, 1, 1, 1])

# 2) Séparation linéaire avec plus de points
X_linear_multiple = np.vstack([
    rng.normal(loc=[1, 1], scale=0.25, size=(60, 2)),
    rng.normal(loc=[2.2, 2.2], scale=0.25, size=(60, 2)),
])
y_linear_multiple = np.array([0] * 60 + [1] * 60)

# 3) XOR : non séparable linéairement
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])

# 4) Cross : problème non linéaire en deux classes
X_cross = rng.uniform(-1, 1, size=(500, 2))
y_cross = ((np.abs(X_cross[:, 0]) < 0.25) | (np.abs(X_cross[:, 1]) < 0.25)).astype(int)

# 5) Multi Linear 3 classes : proche de notre logique one-vs-rest
X_multi_linear = np.vstack([
    rng.normal(loc=[-1.0, -0.5], scale=0.25, size=(80, 2)),
    rng.normal(loc=[1.0, -0.4], scale=0.25, size=(80, 2)),
    rng.normal(loc=[0.0, 1.0], scale=0.25, size=(80, 2)),
])
y_multi_linear = np.array([0] * 80 + [1] * 80 + [2] * 80)

# 6) Multi Cross : multi-classe plus ambigu
X_multi_cross = rng.uniform(-1, 1, size=(700, 2))
radius = np.sqrt(X_multi_cross[:, 0] ** 2 + X_multi_cross[:, 1] ** 2)
y_multi_cross = np.where(radius < 0.35, 0, np.where(X_multi_cross[:, 0] > X_multi_cross[:, 1], 1, 2))

cases = [
    (X_linear_simple, y_linear_simple, "Linear Simple"),
    (X_linear_multiple, y_linear_multiple, "Linear Multiple"),
    (X_xor, y_xor, "XOR"),
    (X_cross, y_cross, "Cross"),
    (X_multi_linear, y_multi_linear, "Multi Linear — 3 classes"),
    (X_multi_cross, y_multi_cross, "Multi Cross"),
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (X_case, y_case, title) in zip(axes.reshape(-1), cases):
    plot_case(ax, X_case, y_case, title)
plt.tight_layout()
plt.show()

## Note sur les cas de régression

Le notebook de cas de tests fourni contient aussi des cas de régression. Ils sont utiles pour comprendre la logique générale du machine learning, mais notre application finale ne prédit pas une valeur continue : elle prédit une classe de bâtiment.

On peut donc les garder comme réflexion pédagogique, mais il faut éviter de les présenter comme une fonctionnalité de l'interface web.

In [ ]:
x = np.linspace(-2, 2, 120)
y_linear = 2 * x + 1
y_nonlinear = np.sin(3 * x) + 0.2 * x

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(x, y_linear, color="#5B8FF9")
axes[0].set_title("Régression linéaire simple — exemple visuel")
axes[0].grid(alpha=0.2)

axes[1].plot(x, y_nonlinear, color="#E8684A")
axes[1].set_title("Régression non linéaire — exemple visuel")
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

## Conclusion

Le projet actuel repose sur un flux clair :

```text
dataset local sans doublon
→ CSV 32×32 / 1 024 features
→ entraînement C++ hors ligne
→ modèles sauvegardés
→ manifest
→ Flask
→ predict_cli.exe
→ prédiction C++
```

La version à présenter dans l'interface web est actuellement **MLP v2**.

RBF v2 est important à garder dans l'explication, car il montre une vraie démarche de diagnostic : RBF v1 échouait à cause d'activations quasi nulles et d'un biais dominant ; RBF v2 corrige le comportement dégénéré avec un sigma et des centres mieux adaptés.